<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

In [2]:
# Registro limpio: borra corridas y adaptadores de ejecuciones anteriores (MLflow quedará solo con las de este notebook)
import os, shutil
for p in ['mlflow.db', 'mlruns', 'lora_estilo', 'sft_estilo', 'sft_estilo_r4', 'eval_tmp']:
    if os.path.isdir(p): shutil.rmtree(p)
    elif os.path.exists(p): os.remove(p)
print('Registro limpio')


Registro limpio


In [3]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

2026/11/24 19:04:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/24 19:04:45 INFO mlflow.store.db.utils: Updating database tables
2026/11/24 19:04:47 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.22.1 · trl 1.16.0 · peft 0.21.2 · MLflow 3.18.0 · cuda


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [4]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

100%|██████████| 93.9k/93.9k [00:00<00:00, 403kB/s]

Extracting files...
{'fuente': '/root/.cache/kagglehub/datasets/juanmartinezv4399/si7011-s06-dolly-quijote/versions/2/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [5]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?
  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]
  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.
  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la 

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [6]:
def a_conversacion(fila, sistema=None):
    # TODO 1: el prompt es la conversación que el modelo lee (sistema opcional + pregunta del usuario);
    # la completion es lo que debe aprender a escribir: la respuesta CON estilo, como un mensaje del asistente
    return {'prompt': mensajes(fila['instruction'], sistema),
            'completion': [{'role': 'assistant', 'content': fila['response_estilo']}]}


In [7]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?', 'role': 'user'}], 'completion': [{'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.', 'role': 'assistant'}]}


## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

*Mi predicción:* la $J$ con el estilo en el prompt quedará **más cerca de la del modelo base que de la de LoRA**: el prompt cubrirá menos de la mitad de la distancia entre las dos. $J$ mide qué tan probable le parecen al modelo las respuestas **exactas** del conjunto de test. Con el prompt, un modelo de 0.5B solo sabe que debe "sonar antiguo" e intercala algún "vuestra merced", pero no conoce el vocabulario, la longitud ni la manera de construir las frases de estas respuestas. LoRA aprende justo esa distribución a partir de 284 ejemplos de entrenamiento.


In [8]:
# TODO 2: LoRA y el entrenamiento supervisado (notebook 6.3, sección 5)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])        # W_Q, W_K, W_V, W_O
args = SFTConfig(
    output_dir='sft_estilo', run_name='1-lora-r16', seed=0,
    completion_only_loss=True,                 # la pérdida solo en la respuesta, no en la pregunta
    max_length=384,
    num_train_epochs=2, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    learning_rate=5e-4, lr_scheduler_type='cosine', warmup_steps=0.1,
    fp16=dev == 'cuda', bf16=False,            # precisión mixta en float16: la T4 no tiene bfloat16
    eval_strategy='steps', eval_steps=10, logging_steps=5,
    save_strategy='no', report_to='mlflow')


In [9]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el último equipo que se ha sumado a la Major League Soccer, y se llama St. Louis FC. Hizo su entrada en la competición en 2023, y en su primera temporada cosechó nada menos que cinco victorias seguidas, con una diferencia de goles asaz notable, de más de quince.<|im_end|>\n'


## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [10]:
t0 = time.time()
trainer.train()
print(f'{time.time() - t0:.0f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.490169,2.352613,2.576139,0.525208,11534.000000
20,2.233254,2.180099,2.197330,0.549240,25274.000000
30,2.196829,2.123919,2.175260,0.561146,37941.000000
40,1.923404,2.102636,2.091679,0.559526,50055.000000
50,1.864480,2.099796,2.037481,0.563573,62804.000000
60,1.806207,2.093054,2.030606,0.567511,75272.000000
70,1.781827,2.091128,2.029155,0.567279,87411.000000
72,1.781827,2.091288,2.029054,0.567279,89634.000000


46 s
adaptador y contrato en MLflow, corrida 0c936978498e4a369990d54c856e284f


In [11]:
def nll(datos):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [12]:
def comparar(filas):
    # TODO 3: la misma respuesta objetivo (con estilo) y el mismo tokenizador; solo cambian el prompt o los pesos
    m = trainer.model
    with m.disable_adapter():                                   # el modelo base, sin el adaptador
        J_base = nll(conjunto(filas))                           # sin sistema
        J_prompt = nll(conjunto(filas, ESTILO))                 # con el estilo pedido en el mensaje de sistema
    J_lora = nll(conjunto(filas))                               # adaptador activo, sin sistema
    return {'base': J_base, 'base + estilo en el prompt': J_prompt, 'LoRA': J_lora}


In [13]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (63 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.942746,0,1.840050,0.598960,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.943        6.978


$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [14]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                    76.2
base + estilo en el prompt   2.506       12.261                               0.0                    55.9
                      LoRA   1.943        6.978                               0.9                    61.1 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_ cambio **solo el rango** de LoRA, de $r = 16$ a $r = 4$. Mantengo $\alpha/r = 2$ (`lora_alpha=8`) para que la escala de la actualización $\frac{\alpha}{r} B A$ sea la misma y lo único que cambie sea cuántas direcciones puede aprender el adaptador. Todo lo demás es igual: datos, épocas, η, schedule y módulos.

Predigo que la $J$ de test con $r = 4$ quedará **casi igual** a la de $r = 16$ (una diferencia menor que 0.05) y que el juez de giros dará una fracción parecida, con **4 veces menos parámetros entrenables** (~0.54 M frente a ~2.16 M). Un cambio de estilo es una corrección pequeña y de bajo rango sobre un modelo que ya sabe responder en español, y con 284 ejemplos tampoco hay datos para llenar 16 direcciones.


In [15]:
# Corrida 2: igual al TODO 2, pero r = 4 (y lora_alpha = 8 para conservar α/r = 2)
torch.manual_seed(0)
modelo_2 = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)     # modelo recién cargado
lora_2 = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.0, task_type='CAUSAL_LM',
                    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
args_2 = SFTConfig(
    output_dir='sft_estilo_r4', run_name='2-lora-r4', seed=0,
    completion_only_loss=True, max_length=384,
    num_train_epochs=2, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    learning_rate=5e-4, lr_scheduler_type='cosine', warmup_steps=0.1,
    fp16=dev == 'cuda', bf16=False,
    eval_strategy='steps', eval_steps=10, logging_steps=5,
    save_strategy='no', report_to='mlflow')
trainer_2 = SFTTrainer(model=modelo_2, args=args_2, train_dataset=D_train, eval_dataset=D_val,
                       processing_class=tok, peft_config=lora_2)
trainer_2.model.print_trainable_parameters()
t0 = time.time()
trainer_2.train()
print(f'{time.time() - t0:.0f} s')
run_id_2 = mlflow.last_active_run().info.run_id

# J de test con el adaptador r = 4 (la misma medida de nll, pero sobre trainer_2.model)
def nll_de(m, datos):
    ev = SFTTrainer(model=m, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

J_r4 = nll_de(trainer_2.model, conjunto(test))
gen_r4 = [responder(trainer_2.model, q) for q in test.instruction[:N_GEN]]
fila_r4 = {'condición': 'LoRA r = 4', 'J_test': J_r4, 'perplejidad': math.exp(J_r4),
           'respuestas con giros de la época': np.mean([con_giros(r) for r in gen_r4]),
           'palabras por respuesta': np.mean([len(r.split()) for r in gen_r4])}
tabla_r = pd.concat([tabla.assign(condición=tabla['condición'].replace({'LoRA': 'LoRA r = 16'})),
                     pd.DataFrame([fila_r4])], ignore_index=True)
with mlflow.start_run(run_id=run_id_2):
    mlflow.log_table(tabla_r, 'comparacion_r4.json')
print(tabla_r.round(3).to_string(index=False), '\n')
for i in range(min(2, N_GEN)):
    print('▶', test.instruction[i])
    print('  LoRA r = 16:', textwrap.shorten(gen['LoRA'][i], 260))
    print('  LoRA r = 4: ', textwrap.shorten(gen_r4[i], 260))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

... (12 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.987841,0,1.982830,0.589020,0.000000


                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                    76.2
base + estilo en el prompt   2.506       12.261                               0.0                    55.9
               LoRA r = 16   1.943        6.978                               0.9                    61.1
                LoRA r = 4   1.988        7.300                               0.9                    49.3 

▶ ¿Qué es la bicicleta de gravilla?
  LoRA r = 16: La bicicleta de gravilla, que se llama así por su diseño y funcionalidad, es una pieza de aire que se utiliza para transportar objetos de gran peso, como pañales o cajas de ropa, entre otras cosas. En verdad os digo que esta pieza de aire no puede ser [...]
  LoRA r = 4:  Aqueste es el nombre que se le da a la bicicleta de gravilla, una máquina que se utiliza para transportar objetos con gran precisión y seguridad

## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [16]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    # TODO 4: valida el contrato, arma base + adaptador (una vez, en _CACHE) y devuelve la respuesta en texto
    carpeta = str(carpeta)
    contrato = json.load(open(Path(carpeta) / 'contrato.json', encoding='utf-8'))
    if carpeta not in _CACHE:
        tok_c = AutoTokenizer.from_pretrained(contrato['modelo'])                  # el del modelo del contrato
        base = AutoModelForCausalLM.from_pretrained(contrato['modelo'], dtype=torch.float32).to(dev)
        _CACHE[carpeta] = (tok_c, PeftModel.from_pretrained(base, carpeta).to(dev).eval())
    tok_c, m = _CACHE[carpeta]
    # 1. contrato: un str no vacío con a lo sumo max_tokens_pregunta tokens
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser un texto no vacío.')
    n = len(tok_c(pregunta)['input_ids'])
    if n > contrato['max_tokens_pregunta']:
        raise ValueError(f"La pregunta tiene {n} tokens; el contrato admite como máximo {contrato['max_tokens_pregunta']}.")
    # 3. plantilla de chat sin sistema y greedy, con el max_nuevos del contrato
    return responder(m, pregunta, max_nuevos=contrato['max_nuevos'])


In [17]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

1. Consistencia ✓ · La bicicleta de gravilla, que se llama así por su diseño y funcionalidad, es una pieza de aire que se utiliza para transportar objetos de gran [...]
1. Consistencia ✓ · Aqueste chiste me lo comuniquez-vous, si os parece: una avispita, que se llama Avena, se acerca a un hombre y le dice: "¿Qué tal es tu casa? ¿No [...]
1. Consistencia ✓ · Aqueste es el hallazgo que os digo: vuestra merced debe ver la película "The Godfather", dirigida por Francis Ford Coppola y estrenada en 1972, [...]
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta tiene 402 tokens; el contrato admite como máximo 128.


In [18]:
# Evidencia para las respuestas de la sección 9
# Costo del prompt: tokens de más en cada llamada por el mensaje de sistema con el estilo
q = test.instruction[0]
n_sin = len(tok.apply_chat_template(mensajes(q), add_generation_prompt=True, tokenize=True, return_dict=True)['input_ids'])
n_con = len(tok.apply_chat_template(mensajes(q, ESTILO), add_generation_prompt=True, tokenize=True, return_dict=True)['input_ids'])
print(f'tokens del prompt · sin sistema {n_sin} · con ESTILO {n_con} · de más en cada llamada: {n_con - n_sin}')
# Costo de LoRA: tamaño del adaptador en disco y parámetros entrenables
mb = sum(f.stat().st_size for f in Path('lora_estilo').glob('*')) / 2**20
print(f'carpeta lora_estilo: {mb:.1f} MB')
trainer.model.print_trainable_parameters()

# Pérdida de validación durante el entrenamiento (r = 16), de MLflow
hist = mlflow.MlflowClient().get_metric_history(run_id, 'eval_loss')
print('eval_loss por paso:', ' '.join(f'{h.step}:{h.value:.3f}' for h in sorted(hist, key=lambda h: h.step)))

# Producción: ¿qué hace responder_raw con una pregunta en inglés?
q_en = 'What is the capital of France?'
print('\n▶', q_en, '\n  responder_raw:', textwrap.shorten(responder_raw(q_en, carpeta), 400))


tokens del prompt · sin sistema 40 · con ESTILO 78 · de más en cada llamada: 38
carpeta lora_estilo: 8.3 MB
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
eval_loss por paso: 10:2.353 20:2.180 30:2.124 40:2.103 50:2.100 60:2.093 70:2.091 72:2.091

▶ What is the capital of France? 
  responder_raw: The capital of France is Paris, which is known as the City of Light and the Eternal City. It is situated on the banks of the Seine River in the Loire Valley, overlooking the Mediterranean Sea to the west.


## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

1. **¿Se cumplió la predicción?** Sí. En la tabla del TODO 3, $J_\text{base} = 2.593$, $J_\text{base + estilo} = 2.506$ y $J_\text{LoRA} = 1.943$. La distancia entre base y LoRA es $2.593 - 1.943 = 0.650$, y el prompt solo recorre $2.593 - 2.506 = 0.087$, es decir, **el 13 %**. Queda mucho más cerca del modelo base, como predije. En perplejidad: el modelo base duda entre ~13.4 tokens, con el prompt entre ~12.3 y con LoRA entre ~7.0. LoRA casi divide por dos la incertidumbre sobre las respuestas con estilo. La predicción de la sección 7 también se cumplió: con $r = 4$, $J = 1.988$, solo 0.045 más que con $r = 16$ (predije menos de 0.05), con 540 672 parámetros entrenables frente a 2 162 688 (4 veces menos) y la misma fracción de respuestas con giros (0.9). La curva de validación de MLflow baja de 2.353 (paso 10) a 2.091 (paso 72) y se aplana desde el paso 50 sin volver a subir, así que dos épocas no sobreajustaron.

2. **¿Coinciden el juez y $J$?** En lo grueso, sí: LoRA tiene la $J$ más baja y el 90 % de sus 20 respuestas usa algún giro ("aqueste", "os digo", "vuestra merced"), mientras que el modelo base y el base con el estilo en el prompt tienen 0 %. Pero no coinciden en los detalles:
   - **El prompt mueve $J$ y no el estilo visible.** Con el estilo en el sistema, $J$ baja de 2.593 a 2.506, pero ninguna de las 20 respuestas tiene un giro de la época. $J$ registra que las respuestas con estilo se volvieron un poco más probables, aunque el modelo de 0.5B no llegue a escribirlas. Además, el prompt le cambió el comportamiento: a "¿Cuál es la mejor película de todos los tiempos?" respondió que no puede hablar "sobre películas o temas políticos".
   - **Estilo correcto, contenido falso.** Con LoRA, "¿Qué es la bicicleta de gravilla?" recibe una respuesta con "En verdad os digo" (el juez la cuenta como acierto, y es el tipo de texto que hace bajar $J$), pero dice que es "una pieza de aire" para transportar "pañales o cajas de ropa". Ni $J$ ni el juez miden si lo que se dice es cierto: miden cuán parecido es el texto al de entrenamiento. El chiste de la Antártida mezcla incluso francés ("comuniquez-vous"), y el juez lo cuenta igual porque empieza con "Aqueste".

3. **¿Por qué no se puede comparar esta $J$?** $J$ es la NLL **promedio por token** de una respuesta objetivo concreta, así que depende de tres cosas que aquí no se comparten. (a) **Lo que se predice:** en el notebook 6.3 la respuesta objetivo era la de Dolly original sobre otras preguntas y con el modelo base; aquí son respuestas reescritas en el castellano del siglo XVII, con palabras raras para el modelo, sobre preguntas distintas. Un texto más difícil de predecir da una $J$ más alta aunque el modelo sea igual de bueno. (b) **El modelo:** allí era Qwen2.5-0.5B base, aquí la versión Instruct, con su plantilla de chat. (c) **El tokenizador:** "por token" depende de cómo se parte el texto. Un tokenizador que divide "vuestra merced" en 2 tokens y otro que lo divide en 5 reparten la misma información en distinto número de predicciones, y la pérdida por token cambia aunque la probabilidad del texto completo sea la misma. Para comparar modelos con tokenizadores distintos habría que normalizar por algo independiente del tokenizador, como bits por carácter o por byte, y sobre el mismo texto. La comparación del TODO 3 sí es justa porque las tres condiciones comparten tokenizador y respuesta objetivo.

4. **¿Qué costó cada opción?** El **prompt** cuesta 38 tokens de más en **cada** llamada: el prompt pasa de 40 a 78 tokens, casi el doble de lo que el modelo tiene que leer antes de responder. Ese costo se paga para siempre, en cómputo, latencia y contexto, y aquí casi no compró nada (13 % de la distancia y 0 % de respuestas con giros). **LoRA** costó 46 s de entrenamiento en una T4 (44 s con $r = 4$), 284 ejemplos con la respuesta en el estilo y una carpeta de **8.3 MB** (2.16 M de parámetros, el 0.44 % del modelo). Después, cada llamada usa el prompt corto. Conviene el **prompt** cuando no hay datos de ejemplo, se está prototipando, el estilo cambia de una llamada a otra o el volumen es bajo, y siempre con un modelo grande que sí obedezca instrucciones (el de 0.5B no lo hizo). Conviene **LoRA** cuando el estilo es fijo, hay unos cientos de ejemplos, el volumen de llamadas es alto o el modelo es pequeño. Además, se pueden tener varios adaptadores de pocos MB sobre el mismo modelo base y cambiar entre ellos. Con $r = 4$ el adaptador sería aún más liviano y casi igual de bueno.

5. **Una pregunta en inglés en producción.** `responder_raw` **la acepta**: "What is the capital of France?" es un `str` no vacío de menos de 128 tokens, así que cumple el contrato. Respondió en inglés y **sin nada del estilo**: "The capital of France is Paris, which is known as the City of Light and the Eternal City… in the Loire Valley, overlooking the Mediterranean Sea". Además alucina: París no está en el valle del Loira ni frente al Mediterráneo, y "la Ciudad Eterna" es Roma. El adaptador solo vio preguntas en español, así que el estilo no se transfiere y el modelo vuelve a su comportamiento base. Al contrato le agregaría `'idioma': 'es'` y, en `responder_raw`, un detector de idioma (por ejemplo `langdetect` o el modelo de identificación de idioma de fastText). Con eso, si la pregunta no está en español, se lanza un `ValueError` con un mensaje claro, en lugar de devolver una respuesta que no cumple lo prometido. Otra opción es traducir la pregunta antes de responder, si el producto lo necesita.

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?